# Data Exploraton: Unwounded ST Woappi Data
Making sure unwounded data is as we expect it to be.

In [ ]:
# combining h5 and parquet into anndata

import scanpy as sc
import pandas as pd
import anndata as ad

base = '/insomnia001/depts/morpheus/users/me2982/data/unwounded_woappi/dropbox_data/008'

# load from h5
adata = sc.read_10x_h5(f'{base}/filtered_feature_bc_matrix_008.h5')

# load spatial coordinates
positions = pd.read_parquet(f'{base}/tissue_positions.parquet')
positions.index = positions['barcode']

# filter to only barcodes in adata and add coordinates
positions = positions.loc[adata.obs_names]
adata.obsm['spatial'] = positions[['pxl_col_in_fullres', 'pxl_row_in_fullres']].values

print(adata)
print("spatial coords first 3:", adata.obsm['spatial'][:3])

adata.write_h5ad("/insomnia001/depts/morpheus/users/me2982/data/unwounded_woappi/data/YWV01_UW.h5ad")

/insomnia001/depts/morpheus/users/me2982/miniconda3/envs/spatialfusion/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/insomnia001/depts/morpheus/users/me2982/miniconda3/envs/spatialfusion/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


AnnData object with n_obs × n_vars = 135179 × 19059
    var: 'gene_ids', 'feature_types', 'genome'
    obsm: 'spatial'
spatial coords first 3: [[17496.48323075 32573.34907785]
 [16281.71187323 32568.92447743]
 [ 9740.1216389  30892.53288372]]


135,179 bins × 19059 genes

In [30]:
import scanpy as sc
import pandas as pd

adata = sc.read_h5ad("/insomnia001/depts/morpheus/users/me2982/data/unwounded_woappi/data/YWV01_UW.h5ad")

/insomnia001/depts/morpheus/users/me2982/miniconda3/envs/spatialfusion/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Checking the qptiff resolution.

In [2]:
import tifffile

wsi = tifffile.imread('/insomnia001/depts/morpheus/users/me2982/data/unwounded_woappi/dropbox_data/B6-298-UW_Scan3.qptiff')
print("WSI shape:", wsi.shape)
print("dtype:", wsi.dtype)

WSI shape: (45360, 25920, 3)
dtype: uint8


In [4]:
print("spatial coords min/max:")
print("x min:", adata.obsm['spatial'][:, 0].min(), "x max:", adata.obsm['spatial'][:, 0].max())
print("y min:", adata.obsm['spatial'][:, 1].min(), "y max:", adata.obsm['spatial'][:, 1].max())
print("WSI dimensions: height=45360, width=25920")

spatial coords min/max:
x min: 7847.622866171074 x max: 21329.712615645352
y min: 27853.75410803128 y max: 41060.945157255745
WSI dimensions: height=45360, width=25920


In [5]:
print(f"total bins in adata: {adata.n_obs}")
print(f"spatial coords shape: {adata.obsm['spatial'].shape}")


total bins in adata: 135179
spatial coords shape: (135179, 2)


## Adding metadata from ST/sc integration analysis (SRSP Lab work)

In [1]:
import pandas as pd

meta = pd.read_csv('/insomnia001/depts/morpheus/users/me2982/data/unwounded_woappi/dropbox_data/008/YWV01_UW_metadata.csv', index_col=0)
print(meta.head())
print(meta.columns.tolist())

                      orig.ident  nCount_Spatial  nFeature_Spatial   Region  \
s_008um_00738_00232-1   YWV01_UW             105                91  Region2   
s_008um_00466_00453-1   YWV01_UW             104                96  Region2   
s_008um_00058_00277-1   YWV01_UW              52                51  Region1   
s_008um_00645_00294-1   YWV01_UW             110               104  Region2   
s_008um_00753_00136-1   YWV01_UW             324               275  Region2   

                       alignment_clusters         spot_class       first_type  \
s_008um_00738_00232-1                   2            singlet  Skeletal Muscle   
s_008um_00466_00453-1                   3             reject       Pericyte I   
s_008um_00058_00277-1                   3             reject            HF II   
s_008um_00645_00294-1                   3  doublet_uncertain  Skeletal Muscle   
s_008um_00753_00136-1                   7    doublet_certain     Papillary II   

                        second_type fi

In [31]:
# filter adata to only bins that have metadata
common_barcodes = list(set(adata.obs_names) & set(meta.index))
adata = adata[common_barcodes].copy()

# add metaclusters to adata.obs
adata.obs = adata.obs.join(meta.loc[adata.obs_names], how='left')

print(adata)
print(adata.obs['metaclusters'].value_counts())

AnnData object with n_obs × n_vars = 113277 × 19059
    obs: 'orig.ident', 'nCount_Spatial', 'nFeature_Spatial', 'Region', 'alignment_clusters', 'spot_class', 'first_type', 'second_type', 'first_class', 'second_class', 'min_score', 'singlet_score', 'conv_all', 'conv_doublet', 'seurat_cluster.sketched', 'seurat_clusters', 'metaclusters', 'AnatomicalSite', 'WoundSite', 'seurat_cluster.projected', 'seurat_cluster.projected.score', 'leverage.score', 'banksy_cluster_0.8', 'banksy_cluster_0.5', 'barcode'
    var: 'gene_ids', 'feature_types', 'genome'
    obsm: 'spatial'
metaclusters
Muscle cells         37541
IFE Keratinocytes    22596
Fibroblasts          20420
HF Keratinocytes     18614
Immune cells          4674
Adipocytes            2477
Sebocytes             1911
Endothelial cells     1577
Pericytes             1534
Unknown               1372
Schwann cells          480
Red Blood Cells         55
Melanocytes             26
Name: count, dtype: int64


/insomnia001/depts/morpheus/users/me2982/miniconda3/envs/spatialfusion/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [32]:
adata.obs = adata.obs.drop(columns=['first_class', 'second_class', 'conv_all', 'conv_doublet'])
adata.write_h5ad('/insomnia001/depts/morpheus/users/me2982/data/unwounded_woappi/data/adata_annotated.h5ad')
print("saved")

saved


# Visualizing spots and patch crops     

In [1]:
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

adata = sc.read_h5ad('/insomnia001/depts/morpheus/users/me2982/data/unwounded_woappi/data/adata_annotated.h5ad')

/insomnia001/depts/morpheus/users/me2982/miniconda3/envs/spatialfusion/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
